# Databricks setup smoke test

Synthetic data only. This notebook creates a dedicated schema/volume, reads the same local fixture, writes and reads a Delta table, transforms load, logs MLflow artifacts, and exports evidence. It does not validate real NYISO data or forecasting methodology.

Run the first two cells, upload the CSV and expected JSON into the printed volume path, then run the remaining cells. Choose a catalog where you can create a schema. No paid compute is required by this notebook.


In [ ]:
import json
import hashlib
import platform
import tempfile
from pathlib import Path
import pandas as pd
import mlflow
from pyspark.sql import functions as F, types as T

# If necessary, replace this with a catalog shown in Catalog Explorer.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "ml_capstone_setup"
VOLUME = "smoke_files"
TABLE = "synthetic_zone_j_smoke"
# Restrict identifiers before inserting them into SQL.
import re
assert all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value)
           for value in [CATALOG, SCHEMA, VOLUME, TABLE]), "Choose simple identifiers"
qualified_schema = f"`{CATALOG}`.`{SCHEMA}`"
qualified_table = f"{qualified_schema}.`{TABLE}`"
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qualified_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {qualified_schema}.`{VOLUME}`")
volume_root = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"Upload synthetic_zone_j_hourly.csv and expected_smoke.json here: {volume_root}")


## Upload checkpoint

In Catalog Explorer, open the catalog printed above → ml_capstone_setup → Volumes → smoke_files → Upload. Upload both files from tests/fixtures in the local project. Then continue below.


In [ ]:
source = volume_root / "synthetic_zone_j_hourly.csv"
expected_path = volume_root / "expected_smoke.json"
assert source.exists() and expected_path.exists(), "Upload both fixture files first"
expected = json.loads(expected_path.read_text())
assert hashlib.sha256(source.read_bytes()).hexdigest() == expected["sample_sha256"]
raw = pd.read_csv(source)
assert len(raw) == expected["rows"]
assert not raw.isna().any().any()
assert not raw.duplicated(["zone", "interval_start_utc"]).any()
assert raw.zone.eq("J").all()
start = pd.to_datetime(raw.interval_start_utc, utc=True)
end = pd.to_datetime(raw.interval_end_utc, utc=True)
assert (end - start).dt.total_seconds().eq(3600).all()
assert start.diff().iloc[1:].dt.total_seconds().eq(3600).all()
local = start.dt.tz_convert("America/New_York")
assert local.dt.strftime("%Y-%m-%d %H:%M").eq("2024-11-03 01:00").sum() == 2
schema = T.StructType([
    T.StructField("interval_start_utc", T.TimestampType(), False),
    T.StructField("interval_end_utc", T.TimestampType(), False),
    T.StructField("zone", T.StringType(), False),
    T.StructField("load_mw", T.DoubleType(), False),
    T.StructField("temperature_c", T.DoubleType(), False),
])
frame = (spark.read.option("header", True).option("mode", "FAILFAST")
         .schema(schema).csv(str(source)))
transformed = frame.withColumn("load_gw", F.col("load_mw") / F.lit(1000.0))
# Overwrite only this dedicated synthetic setup table; never a research table.
transformed.write.format("delta").mode("overwrite").saveAsTable(qualified_table)
readback = spark.table(qualified_table)
assert readback.count() == expected["rows"]
assert readback.exceptAll(transformed).count() == 0
assert transformed.exceptAll(readback).count() == 0
assert readback.filter(F.col("load_gw").isNull()).count() == 0
assert readback.filter(F.abs(F.col("load_gw") - F.col("load_mw") / 1000) > 1e-12).count() == 0
summary_row = readback.agg(F.sum("load_mw").alias("total"),
                         F.avg("load_mw").alias("mean"),
                         F.max("load_mw").alias("maximum")).first()
summary = {"fixture": expected["fixture"], "rows": readback.count(),
           "total_load_mw": float(summary_row.total),
           "mean_load_mw": float(summary_row.mean),
           "max_load_mw": float(summary_row.maximum),
           "sample_sha256": hashlib.sha256(source.read_bytes()).hexdigest()}
assert summary == expected, (summary, expected)
display(readback.orderBy("interval_start_utc"))
print(json.dumps(summary, indent=2))


In [ ]:
# In an imported notebook, Databricks creates a notebook experiment by default.
mlflow.set_tracking_uri("databricks")
with tempfile.TemporaryDirectory() as temp:
    temp_path = Path(temp)
    (temp_path / "smoke_summary.json").write_text(json.dumps(summary, indent=2))
    (temp_path / "runtime.json").write_text(json.dumps({
        "python": platform.python_version(), "pandas": pd.__version__,
        "mlflow": mlflow.__version__, "spark": spark.version}, indent=2))
    with mlflow.start_run(run_name="synthetic_setup_smoke") as run:
        mlflow.set_tag("data_kind", "synthetic_setup_only")
        mlflow.log_param("sample_sha256", summary["sample_sha256"])
        mlflow.log_param("delta_table", f"{CATALOG}.{SCHEMA}.{TABLE}")
        mlflow.log_metric("row_count", summary["rows"])
        mlflow.log_metric("mean_load_mw", summary["mean_load_mw"])
        mlflow.log_artifacts(str(temp_path))
        run_id = run.info.run_id
    recorded = mlflow.get_run(run_id)
    assert recorded.data.metrics["row_count"] == expected["rows"]
    restored = Path(mlflow.artifacts.download_artifacts(
        run_id=run_id, artifact_path="smoke_summary.json", dst_path=temp))
    assert json.loads(restored.read_text()) == expected

evidence = {**summary, "status": "passed", "mlflow_run_id": run_id,
            "experiment_id": recorded.info.experiment_id,
            "delta_table": f"{CATALOG}.{SCHEMA}.{TABLE}",
            "executed_at_utc": pd.Timestamp.now(tz="UTC").isoformat()}
(volume_root / "databricks_smoke.json").write_text(json.dumps(evidence, indent=2))
readback.orderBy("interval_start_utc").toPandas().to_csv(
    volume_root / "databricks_smoke_result.csv", index=False)
print(json.dumps(evidence, indent=2))
print(f"Download databricks_smoke.json and databricks_smoke_result.csv from {volume_root}")
print("Export this notebook with outputs as .ipynb. Remote verification is complete only after download checks.")
